In [1]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

from llama_index.llms.openai import OpenAI

# Grader LLM - can be same or different model than the agent
grader_llm = OpenAI(model="gpt-4o-mini", temperature=0)

print("Grader LLM ready:", grader_llm.model)

Grader LLM ready: gpt-4o-mini


In [2]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

chroma_client     = chromadb.PersistentClient(path="c:/Users/aarus/Context_core/utils/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("contextcore")
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=3)

print("Index loaded")

Index loaded


In [3]:
test_query = "Fed raises interest rates by 0.25%. What does this mean for tech stocks and inflation?"

# Retrieve chunks (same as agent did)
nodes = retriever.retrieve(test_query)

retrieved_chunks = []
for node in nodes:
    retrieved_chunks.append({
        "source": node.metadata.get("source", "unknown"),
        "published_at": node.metadata.get("published_at", "unknown"),
        "text": node.text[:500]
    })

# Domain classification (from Day 2 output)
domain_classification = "macro, finance, tech, cross-domain"

# The agent's answer from Day 2
agent_answer = """The Fed's decision to raise interest rates by 0.25% typically leads to higher borrowing costs for companies, including those in the tech sector. This can negatively impact tech stocks, especially those that rely heavily on debt for expansion. However, the tech sector has shown resilience, as evidenced by recent gains in major indices like the S&P 500 and Nasdaq 100.

On the inflation front, the rate hike is a response to rising inflation, which has recently surpassed 4%. Higher interest rates are intended to curb inflation by reducing consumer spending and investment. Therefore, while the rate increase may pose challenges for tech stocks, it is also a necessary measure to address ongoing inflationary pressures."""

print(f"Query: {test_query}")
print(f"Domain: {domain_classification}")
print(f"Chunks retrieved: {len(retrieved_chunks)}")
for c in retrieved_chunks:
    print(f"  - [{c['source']}] {c['text'][:80]}...")
print(f"\nAnswer length: {len(agent_answer)} chars")

Query: Fed raises interest rates by 0.25%. What does this mean for tech stocks and inflation?
Domain: macro, finance, tech, cross-domain
Chunks retrieved: 3
  - [FRED] FRED Data: Federal Funds Rate

In 2024-06, the Federal Funds Rate was 5.33.
In 2...
  - [FRED] FRED Data: 10-Year Breakeven Inflation Rate

In 2026-05, the 10-Year Breakeven I...
  - [Wikipedia] Wikipedia: Inflation

In economics, inflation is an increase in the average pric...

Answer length: 720 chars


In [4]:
GRADER_PROMPT = """You are a strict evaluation agent. Your job is to grade an AI-generated answer 
against the retrieved source chunks it was based on. You must ONLY use the provided chunks as ground truth — 
do NOT use your own knowledge to judge factual correctness.

ORIGINAL QUERY:
{query}

DOMAIN CLASSIFICATION:
{domain}

RETRIEVED CHUNKS (the only allowed sources of truth):
{chunks}

AGENT'S ANSWER:
{answer}

Score the answer on these 6 metrics, each from 0.0 to 1.0:

1. faithfulness: Are claims in the answer supported by the retrieved chunks? 
   Penalize claims that appear nowhere in the chunks.

2. context_relevance: Are the retrieved chunks actually relevant to the query? 
   (This grades the RETRIEVAL step, not the answer.)

3. causal_validity: For any cause-effect reasoning in the answer 
   (e.g. "X leads to Y leads to Z"), does each step follow logically? 
   Score 1.0 if the chain is plausible economic/financial reasoning, 
   lower if steps are missing or illogical.

4. cross_domain_coverage: The domain classification lists certain domains. 
   Does the answer address ALL of them, or does it ignore some?

5. completeness: Does the answer address all parts of the query, 
   or does it skip parts of a multi-part question?

6. source_recency: For any claims about "what happened" or current events, 
   are they based on recent sources (news) rather than only old/static 
   background sources (Wikipedia)?

Return ONLY valid JSON, no other text, in this exact format:
{{
  "faithfulness": 0.0,
  "context_relevance": 0.0,
  "causal_validity": 0.0,
  "cross_domain_coverage": 0.0,
  "completeness": 0.0,
  "source_recency": 0.0,
  "flagged_claims": ["list any claims not supported by chunks"],
  "notes": "1-2 sentence summary of main issues, if any"
}}
"""

print("Grader prompt template ready")
print(f"Length: {len(GRADER_PROMPT)} chars")

Grader prompt template ready
Length: 1767 chars


In [6]:
# Add the news chunks that the agent actually used in Day 2
news_chunks = [
    {"source": "NewsAPI", "published_at": "2026-05-29", "text": "The War Against Inflation Would Nearly Be Over If Not for Trump - inflation surpassed 4%"},
    {"source": "NewsAPI", "published_at": "2026-06-08", "text": "Last week's stock plunge was a 'healthy reset' - S&P 500 and Nasdaq 100 gains"},
]

all_chunks = retrieved_chunks + news_chunks

chunks_text = "\n\n".join([
    f"[Chunk {i+1} - Source: {c['source']}, Date: {c.get('published_at','unknown')}]\n{c['text']}"
    for i, c in enumerate(all_chunks)
])

filled_prompt = GRADER_PROMPT.format(
    query=test_query,
    domain=domain_classification,
    chunks=chunks_text,
    answer=agent_answer
)

response = grader_llm.complete(filled_prompt)
print(response.text)

{
  "faithfulness": 0.5,
  "context_relevance": 0.5,
  "causal_validity": 1.0,
  "cross_domain_coverage": 0.5,
  "completeness": 0.5,
  "source_recency": 0.5,
  "flagged_claims": ["inflation has recently surpassed 4%", "the tech sector has shown resilience, as evidenced by recent gains in major indices like the S&P 500 and Nasdaq 100"],
  "notes": "The answer contains claims about inflation and tech stock performance that are not directly supported by the retrieved chunks. Additionally, it does not fully address all domains in the query."
}


In [7]:
# A deliberately hallucinated answer — facts that contradict/aren't in our chunks
bad_answer = """The Fed cut interest rates by 0.25% to boost the economy. 
This caused tech stocks to crash by 15% overnight. Inflation dropped to 1.2%, 
the lowest in a decade, as a direct result of this rate cut. 
The unemployment rate spiked to 8% following this announcement."""

filled_prompt = GRADER_PROMPT.format(
    query=test_query,
    domain=domain_classification,
    chunks=chunks_text,   # same chunks as before
    answer=bad_answer
)

response = grader_llm.complete(filled_prompt)
print(response.text)

{
  "faithfulness": 0.0,
  "context_relevance": 0.0,
  "causal_validity": 0.0,
  "cross_domain_coverage": 0.0,
  "completeness": 0.0,
  "source_recency": 0.0,
  "flagged_claims": [
    "The Fed cut interest rates by 0.25%",
    "This caused tech stocks to crash by 15% overnight",
    "Inflation dropped to 1.2%",
    "The unemployment rate spiked to 8%"
  ],
  "notes": "The answer contains multiple claims that are not supported by the retrieved chunks, including incorrect information about interest rate changes, stock performance, inflation rates, and unemployment rates. Additionally, the retrieved chunks do not provide relevant context for the claims made."
}


In [8]:
import json

def grade_answer(query: str, domain: str, chunks: list, answer: str) -> dict:
    """
    Grades an agent's answer against retrieved chunks across 6 metrics.
    
    Args:
        query: the original user headline/question
        domain: domain classification string
        chunks: list of dicts with 'source', 'published_at', 'text'
        answer: the agent's final answer text
    
    Returns:
        dict with 6 scores, flagged_claims, and notes
    """
    chunks_text = "\n\n".join([
        f"[Chunk {i+1} - Source: {c['source']}, Date: {c.get('published_at','unknown')}]\n{c['text']}"
        for i, c in enumerate(chunks)
    ])
    
    prompt = GRADER_PROMPT.format(
        query=query,
        domain=domain,
        chunks=chunks_text,
        answer=answer
    )
    
    response = grader_llm.complete(prompt)
    
    # Strip markdown code fences if present
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].replace("json", "", 1).strip()
    
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "Failed to parse grader output", "raw": raw}


# Test the function
result = grade_answer(test_query, domain_classification, all_chunks, agent_answer)
print(json.dumps(result, indent=2))

{
  "faithfulness": 0.5,
  "context_relevance": 0.5,
  "causal_validity": 1.0,
  "cross_domain_coverage": 1.0,
  "completeness": 0.5,
  "source_recency": 0.5,
  "flagged_claims": [
    "inflation has recently surpassed 4%",
    "the tech sector has shown resilience, as evidenced by recent gains in major indices like the S&P 500 and Nasdaq 100"
  ],
  "notes": "The answer includes claims about inflation and tech stock performance that are not directly supported by the retrieved chunks. Additionally, while it addresses both tech stocks and inflation, it lacks specific data from the chunks to substantiate its claims."
}
